# 05 · Previsões com o modelo vigente e teste integrado do ciclo

Passo **10 do fluxo único**. O modelo vigente é carregado **junto com o pré-processamento e o conjunto de features** correspondentes (é um único pipeline serializado, com hash conferido contra o manifesto) e gera previsões para registros **não utilizados na escolha do modelo**: o conjunto de **teste** reservado (avaliação final) e URLs de exemplo.

Depois, verificações automáticas confirmam que neste ciclo a votação foi refeita, os dois algoritmos foram treinados e o vigente foi promovido ou mantido automaticamente; e comparamos os ciclos executados até agora.

In [ ]:
import sys
from pathlib import Path

# Caminhos consistentes: localiza a raiz do projeto (pasta com config/config.yaml)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import preparar_notebook

cfg = preparar_notebook()  # muda o diretório de trabalho para a raiz do projeto

import json

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 80, "figure.figsize": (9, 4)})

from src import ciclo, ingestao, modelagem, promocao, rastreamento, relatorio
from src import preprocessamento as pp
from src.config import ler_json, salvar_json

ctx = ciclo.carregar_ciclo(exigir=["01_etl", "02_votacao", "03_treino", "04_promocao"])
EVID = ciclo.dir_evidencias(ctx)
classes = cfg["dados"]["classes"]
manifesto, vigente = promocao.carregar_vigente()
assert vigente is not None, "não há modelo vigente aprovado para gerar previsões"
v = manifesto["vigente"]
print(f"cycle_id = {ctx['cycle_id']} | vigente = {v['versao']} ({v['algoritmo']}, promovido no ciclo {v['cycle_id']}, data_version {v['data_version']})")
print(vigente)

## Previsões no conjunto de teste reservado (avaliação final)

In [ ]:
base = pd.read_parquet(Path(cfg["dados"]["dir_processed"]) / "base_features.parquet")
teste = base[base["particao"] == "teste"].reset_index(drop=True)
proba = vigente.prever_proba(teste["url"])                    # a partir da URL crua: extração + pré-processamento + features
y_teste = modelagem.codificar_alvo(teste["type"], classes)
m_teste = modelagem.calcular_metricas(y_teste, proba.to_numpy(), classes)
display(pd.Series(m_teste, name="teste").round(4).to_frame().T)
display(modelagem.matriz_confusao(y_teste, proba.to_numpy(), classes))

previsoes = pd.DataFrame({"url": teste["url"], "real": teste["type"], "previsto": proba.idxmax(axis=1),
                          "confianca": proba.max(axis=1).round(4)}).join(proba.round(4).add_prefix("p_"))
previsoes.to_csv(EVID / "previsoes_teste.csv.gz", index=False, compression="gzip")
previsoes.head(300).to_csv(EVID / "previsoes_teste_amostra.csv", index=False)
display(previsoes.head(10))

## Previsões para URLs novas (uso do modelo)

URLs fora da base. Atenção: o modelo vê só o texto da URL; domínios curtos sem caminho (como `google.com`) são justamente a região em que benign e phishing mais se sobrepõem nesta base, então podem ser classificados de forma errada — limitação discutida no relatório.

In [ ]:
exemplos = ["github.com/pedrosantos2/n3_cybersecurity", "https://www.google.com", "http://drive-google-com.verify-session.com/5e858109c170e2415e/",
            "http://192.168.10.5:8080/bins/payload.arm7", "bit.ly/3xYz9Ab", "http://www.example.com/index.php?option=com_content&view=article&id=7"]
saida = vigente.prever_proba(exemplos).round(4)
saida.insert(0, "previsto", vigente.prever(exemplos))
saida.insert(0, "url", exemplos)
display(saida)

## Teste integrado do ciclo

In [ ]:
runs = rastreamento.runs_do_ciclo(cfg, ctx["cycle_id"])
etapas = runs["tags.etapa"].value_counts().to_dict()
treinos = runs[runs["tags.etapa"] == "03_treino"]
run_prom = runs[runs["tags.etapa"] == "04_promocao"]
votacao = ler_json(EVID / "votacao_features.json")

# Consistência: o modelo registrado no MLflow (alias vigente) prevê igual ao bundle versionado
nome, alias = cfg["mlflow"]["modelo_registrado"], cfg["mlflow"]["alias_vigente"]
amostra = teste.head(300)
pyfunc = mlflow.pyfunc.load_model(f"models:/{nome}@{alias}")
prev_mlflow = np.asarray(pyfunc.predict(amostra[pp.COLUNAS_FEATURES]))
prev_bundle = vigente.pipeline.predict(amostra[pp.COLUNAS_FEATURES])

verificacoes = [
    {"verificacao": "votação refeita neste ciclo", "ok": etapas.get("02_votacao", 0) == 1 and votacao["cycle_id"] == ctx["cycle_id"]},
    {"verificacao": "dois algoritmos distintos treinados neste ciclo", "ok": len(treinos) == 2 and treinos["tags.algoritmo"].nunique() == 2},
    {"verificacao": "decisão automática registrada (promover/manter)", "ok": len(run_prom) == 1 and run_prom["tags.decisao"].iloc[0] in ("promover", "manter")
                                                                            and manifesto["ultima_decisao"]["cycle_id"] == ctx["cycle_id"]},
    {"verificacao": "mesmo cycle_id/data_version em todas as etapas", "ok": set(runs["tags.data_version"].dropna()) == {ctx["data_version"]}},
    {"verificacao": "features do bundle = features do manifesto", "ok": vigente.features == v["features"]},
    {"verificacao": "MLflow (alias vigente) e bundle prevêem igual", "ok": bool(np.array_equal(prev_mlflow.astype(int), prev_bundle.astype(int)))},
    {"verificacao": "previsões geradas para registros não usados na escolha", "ok": len(previsoes) == len(teste) > 0},
]
verif = pd.DataFrame(verificacoes)
display(verif)
assert verif["ok"].all(), f"Falha no teste integrado: {verif[~verif['ok']]['verificacao'].tolist()}"

## Comparação entre ciclos (o que mudou)

In [ ]:
historico = promocao.ler_historico()
registro = ingestao.carregar_registro(cfg)
lotes = pd.DataFrame(registro["lotes"])
novos_por_ciclo = lotes[lotes["status"] == "incorporado"].groupby("incorporado_no_ciclo")["arquivo"].apply(lambda s: ", ".join(Path(a).stem for a in s))
comparacao = historico.set_index("cycle_id").join(novos_por_ciclo.rename("lotes_incorporados"))
mudancas = []
for cid in comparacao.index:
    vot = ler_json(Path("evidencias") / cid / "votacao_features.json") or {}
    mudancas.append({"cycle_id": cid, "n_features": len(vot.get("eleitas", [])),
                     "features_entraram": ",".join(vot.get("mudancas", {}).get("entraram", [])) if vot.get("mudancas", {}).get("ciclo_anterior") else "(primeiro ciclo)",
                     "features_sairam": ",".join(vot.get("mudancas", {}).get("sairam", []))})
comparacao = comparacao.join(pd.DataFrame(mudancas).set_index("cycle_id"))
cols = ["data_version", "gatilho", "lotes_incorporados", "n_features", "features_entraram", "features_sairam", "algoritmo_vencedor",
        "cv_vencedor", "metrica_desafiante_validacao", "metrica_vigente_validacao", "delta", "decisao", "versao_vigente_apos"]
display(comparacao[cols])
comparacao[cols].to_csv(EVID / "comparacao_ciclos.csv")

if len(historico) > 1:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(historico["cycle_id"].str[:5], historico["cv_vencedor"], marker="o", label="CV do vencedor do ciclo")
    ax.plot(historico["cycle_id"].str[:5], historico["metrica_desafiante_validacao"], marker="s", label="desafiante (validação)")
    ax.plot(historico["cycle_id"].str[:5], historico["metrica_vigente_validacao"], marker="^", label="vigente (validação)")
    for i, d in enumerate(historico["decisao"]):
        ax.annotate(d, (i, historico["metrica_desafiante_validacao"].iloc[i]), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
    ax.set_ylabel(cfg["avaliacao"]["metrica_principal"]); ax.set_title("Evolução entre ciclos"); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(EVID / "evolucao_ciclos.png"); plt.show()

## Registro no MLflow e encerramento do ciclo

In [ ]:
salvar_json({"cycle_id": ctx["cycle_id"], "metricas_teste": m_teste, "verificacoes": verificacoes, "vigente": v["versao"]},
            EVID / "previsoes_verificacoes.json")
with rastreamento.run_etapa(ctx, cfg, "05_previsoes", "05_previsoes", tags={"versao_vigente": v["versao"]}) as run:
    mlflow.log_params({"versao_vigente": v["versao"], "algoritmo_vigente": v["algoritmo"], "run_id_vigente": v["run_id"], "n_teste": len(teste)})
    mlflow.log_metrics({f"teste_{k}": val for k, val in m_teste.items()})
    for nome_arq in ["previsoes_teste.csv.gz", "previsoes_verificacoes.json", "comparacao_ciclos.csv"]:
        mlflow.log_artifact(str(EVID / nome_arq))
    run_id = run.info.run_id

rastreamento.atualizar_run_pai(ctx, tags={"status_ciclo": "concluido"}, metricas={f"teste_{cfg['avaliacao']['metrica_principal']}_vigente": m_teste[cfg["avaliacao"]["metrica_principal"]]})
ctx = ciclo.registrar_etapa(ctx, "05_previsoes", run_id=run_id, metricas_teste=m_teste)
rastreamento.encerrar_run_pai(ctx, "FINISHED")
resumo = relatorio.salvar_resumo_ciclo(ctx)
display(Markdown(resumo))